In [1]:
import sys

from pyspark.sql import SparkSession
from pyspark.sql import functions as F


PROJECT_ROOT = "/workspace/nyc-building-risk"
COMMON_PATH = f"{PROJECT_ROOT}/spark/common"

if COMMON_PATH not in sys.path:
    sys.path.insert(0, COMMON_PATH)

from minio_config import configure_minio, minio_path


spark = (
    SparkSession.builder
    .appName("NYC Building Risk - Gold Risk Features")
    .master("local[2]")
    .config("spark.driver.memory", "2g")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.session.timeZone", "UTC")
    .config(
        "spark.hadoop.fs.s3a.impl",
        "org.apache.hadoop.fs.s3a.S3AFileSystem"
    )
    .getOrCreate()
)

configure_minio(spark)

spark.sparkContext.setLogLevel("WARN")

print("Spark:", spark.version)
print("Test:", spark.range(1).count())

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/05 19:56:12 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/09/05 19:56:13 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Spark: 3.4.0
Test: 1


In [2]:
building_identity_df = spark.read.parquet(
    minio_path("gold/building_identity/final")
)

nyc311_df = spark.read.parquet(
    minio_path("silver/nyc_311")
)

hpd_df = spark.read.parquet(
    minio_path("silver/hpd")
)

dob_df = spark.read.parquet(
    minio_path("silver/dob")
)

print("Building Risk sources loaded")

26/09/05 19:56:33 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


Building Risk sources loaded


In [3]:
print("=== NYC 311 ===")
nyc311_df.printSchema()

print("\n=== HPD CLASS VALUES ===")
hpd_df \
    .groupBy("class") \
    .count() \
    .orderBy(F.col("count").desc()) \
    .show(30, truncate=False)

print("\n=== DOB STATUS VALUES ===")
dob_df \
    .groupBy("violation_status") \
    .count() \
    .orderBy(F.col("count").desc()) \
    .show(30, truncate=False)

=== NYC 311 ===
root
 |-- unique_key: string (nullable = true)
 |-- created_date: timestamp (nullable = true)
 |-- closed_date: timestamp (nullable = true)
 |-- resolution_action_updated_date: timestamp (nullable = true)
 |-- agency: string (nullable = true)
 |-- agency_name: string (nullable = true)
 |-- complaint_type: string (nullable = true)
 |-- descriptor: string (nullable = true)
 |-- descriptor_2: string (nullable = true)
 |-- status: string (nullable = true)
 |-- incident_address: string (nullable = true)
 |-- street_name: string (nullable = true)
 |-- incident_zip: string (nullable = true)
 |-- borough: string (nullable = true)
 |-- city: string (nullable = true)
 |-- bbl: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- community_board: string (nullable = true)
 |-- council_district: string (nullable = true)
 |-- location_type: string (nullable = true)
 |-- open_data_channel_type: string (nullable = true)
 |-- cre

+-----+------+
|class|count |
+-----+------+
|B    |370033|
|C    |273694|
|A    |214695|
|I    |68886 |
+-----+------+


=== DOB STATUS VALUES ===


+--------------------------+------+
|violation_status          |count |
+--------------------------+------+
|ACTIVE                    |127878|
|DISMISSED                 |17309 |
|DISPUTED SUCCESSFULLY     |2543  |
|WAIVED - PENDING DISMISSAL|412   |
|null                      |396   |
|PENDING DISMISSAL         |77    |
|CURED                     |73    |
+--------------------------+------+



In [4]:
from pyspark.sql import functions as F


# ==================================================
# RISK AS-OF DATE
# ==================================================

risk_as_of_date = (
    nyc311_df
    .select(
        F.max(
            F.to_date("created_date")
        ).alias("max_date")
    )
    .first()["max_date"]
)

print("Risk as-of date:", risk_as_of_date)

Risk as-of date: 2026-08-23


In [5]:
# ==================================================
# HPD RISK FEATURES PER BIN
# ==================================================

hpd_features = (
    hpd_df

    .filter(
        F.col("bin").isNotNull()
    )

    .groupBy("bin")

    .agg(
        F.count("*").alias(
            "hpd_total_violations"
        ),

        F.sum(
            F.when(
                F.col("class") == "A",
                1
            ).otherwise(0)
        ).alias("hpd_class_a"),

        F.sum(
            F.when(
                F.col("class") == "B",
                1
            ).otherwise(0)
        ).alias("hpd_class_b"),

        F.sum(
            F.when(
                F.col("class") == "C",
                1
            ).otherwise(0)
        ).alias("hpd_class_c"),

        F.sum(
            F.when(
                F.col("class") == "I",
                1
            ).otherwise(0)
        ).alias("hpd_class_i"),

        F.sum(
            F.when(
                F.to_date("inspectiondate")
                >=
                F.date_sub(
                    F.lit(risk_as_of_date),
                    30
                ),
                1
            ).otherwise(0)
        ).alias("hpd_last_30_days"),

        F.sum(
            F.when(
                F.to_date("inspectiondate")
                >=
                F.date_sub(
                    F.lit(risk_as_of_date),
                    90
                ),
                1
            ).otherwise(0)
        ).alias("hpd_last_90_days"),

        F.sum(
            F.when(
                F.to_date("inspectiondate")
                >=
                F.date_sub(
                    F.lit(risk_as_of_date),
                    365
                ),
                1
            ).otherwise(0)
        ).alias("hpd_last_365_days"),

        F.max(
            "inspectiondate"
        ).alias("hpd_latest_violation_date")
    )
)

In [6]:
# ==================================================
# DOB RISK FEATURES PER BIN
# ==================================================

dob_features = (
    dob_df

    .filter(
        F.col("bin").isNotNull()
    )

    .groupBy("bin")

    .agg(
        F.count("*").alias(
            "dob_total_violations"
        ),

        F.sum(
            F.when(
                F.col("violation_status") == "ACTIVE",
                1
            ).otherwise(0)
        ).alias("dob_active_violations"),

        F.sum(
            F.when(
                F.to_date("violation_issue_date")
                >=
                F.date_sub(
                    F.lit(risk_as_of_date),
                    30
                ),
                1
            ).otherwise(0)
        ).alias("dob_last_30_days"),

        F.sum(
            F.when(
                F.to_date("violation_issue_date")
                >=
                F.date_sub(
                    F.lit(risk_as_of_date),
                    90
                ),
                1
            ).otherwise(0)
        ).alias("dob_last_90_days"),

        F.sum(
            F.when(
                F.to_date("violation_issue_date")
                >=
                F.date_sub(
                    F.lit(risk_as_of_date),
                    365
                ),
                1
            ).otherwise(0)
        ).alias("dob_last_365_days"),

        F.max(
            "violation_issue_date"
        ).alias("dob_latest_violation_date")
    )
)

In [7]:
HPD_FEATURES_PATH = minio_path(
    "gold/building_risk/intermediate/hpd_features"
)

DOB_FEATURES_PATH = minio_path(
    "gold/building_risk/intermediate/dob_features"
)


(
    hpd_features
    .write
    .mode("overwrite")
    .parquet(HPD_FEATURES_PATH)
)

(
    dob_features
    .write
    .mode("overwrite")
    .parquet(DOB_FEATURES_PATH)
)

print("HPD and DOB risk features saved successfully")

HPD and DOB risk features saved successfully


In [8]:
hpd_features_check = spark.read.parquet(
    HPD_FEATURES_PATH
)

dob_features_check = spark.read.parquet(
    DOB_FEATURES_PATH
)

print(
    "HPD buildings:",
    hpd_features_check.count()
)

print(
    "DOB buildings:",
    dob_features_check.count()
)

hpd_features_check.show(
    5,
    truncate=False
)

dob_features_check.show(
    5,
    truncate=False
)

HPD buildings: 124889
DOB buildings: 106957
+-------+--------------------+-----------+-----------+-----------+-----------+----------------+----------------+-----------------+-------------------------+
|bin    |hpd_total_violations|hpd_class_a|hpd_class_b|hpd_class_c|hpd_class_i|hpd_last_30_days|hpd_last_90_days|hpd_last_365_days|hpd_latest_violation_date|
+-------+--------------------+-----------+-----------+-----------+-----------+----------------+----------------+-----------------+-------------------------+
|3341749|37                  |13         |17         |7          |0          |2               |2               |37               |2026-08-02 00:00:00      |
|3027834|16                  |0          |4          |12         |0          |0               |0               |16               |2026-03-28 00:00:00      |
|4433888|42                  |6          |19         |17         |0          |0               |9               |42               |2026-07-21 00:00:00      |
|3110485|174  

In [9]:
from pyspark.sql import functions as F


# ==================================================
# BUILD ALL KNOWN BBL -> BIN RELATIONSHIPS
# ==================================================

building_bbl_ref = (
    building_identity_df

    .withColumn(
        "all_bbls",
        F.array_distinct(
            F.concat(
                F.array(
                    F.col("current_bbl"),
                    F.col("resolved_bbl")
                ),
                F.col("bbl_aliases")
            )
        )
    )

    .select(
        "bin",
        F.explode_outer("all_bbls").alias("bbl")
    )

    .filter(
        F.col("bbl").isNotNull()
        & (F.trim(F.col("bbl")) != "")
    )

    .dropDuplicates(
        ["bin", "bbl"]
    )
)

In [10]:
# ==================================================
# COUNT BINS PER BBL
# ==================================================

bbl_bin_stats = (
    building_bbl_ref

    .groupBy("bbl")

    .agg(
        F.countDistinct("bin").alias("bin_count"),

        F.first("bin").alias("single_bin")
    )
)

In [11]:
# ==================================================
# UNIQUE BBL -> BIN ONLY
# ==================================================

unique_bbl_to_bin = (
    bbl_bin_stats

    .filter(
        F.col("bin_count") == 1
    )

    .select(
        "bbl",
        F.col("single_bin").alias("resolved_bin")
    )
)

In [12]:
BBL_BIN_REF_PATH = minio_path(
    "gold/building_risk/intermediate/bbl_to_bin_reference"
)

(
    unique_bbl_to_bin
    .write
    .mode("overwrite")
    .parquet(BBL_BIN_REF_PATH)
)

print("BBL -> BIN reference saved")

BBL -> BIN reference saved


In [13]:
unique_bbl_check = spark.read.parquet(
    BBL_BIN_REF_PATH
)

print(
    "Unique BBL -> BIN mappings:",
    unique_bbl_check.count()
)

print(
    "BBLs with multiple BINs:",
    bbl_bin_stats
    .filter(F.col("bin_count") > 1)
    .count()
)

Unique BBL -> BIN mappings: 160915
BBLs with multiple BINs: 11760


In [14]:
311_bbl_resolved = (
    nyc311_df

    .filter(
        F.col("bbl").isNotNull()
    )

    .join(
        unique_bbl_check,
        on="bbl",
        how="inner"
    )
)

print(
    "311 resolved by unique BBL:",
    311_bbl_resolved.count()
)

print(
    "311 total:",
    nyc311_df.count()
)

SyntaxError: invalid decimal literal (3342010950.py, line 1)

In [15]:
from pyspark.sql import functions as F


# ==================================================
# 1. BUILD ALL KNOWN BBL -> BIN RELATIONSHIPS
# ==================================================

building_bbl_ref = (
    building_identity_df

    .withColumn(
        "all_bbls",
        F.array_distinct(
            F.concat(
                F.array(
                    F.col("current_bbl"),
                    F.col("resolved_bbl")
                ),
                F.col("bbl_aliases")
            )
        )
    )

    .select(
        "bin",
        F.explode_outer("all_bbls").alias("bbl")
    )

    .filter(
        F.col("bbl").isNotNull()
        & (F.trim(F.col("bbl")) != "")
    )

    .dropDuplicates(
        ["bin", "bbl"]
    )
)


# ==================================================
# 2. COUNT HOW MANY BINs EXIST FOR EACH BBL
# ==================================================

bbl_bin_stats = (
    building_bbl_ref

    .groupBy("bbl")

    .agg(
        F.countDistinct("bin").alias("bin_count"),
        F.first("bin").alias("single_bin")
    )
)


# ==================================================
# 3. KEEP ONLY SAFE / UNIQUE BBL -> BIN MAPPINGS
# ==================================================

unique_bbl_to_bin = (
    bbl_bin_stats

    .filter(
        F.col("bin_count") == 1
    )

    .select(
        "bbl",
        F.col("single_bin").alias("resolved_bin")
    )
)


# ==================================================
# 4. SAVE BBL -> BIN REFERENCE TO MINIO
# ==================================================

BBL_BIN_REF_PATH = minio_path(
    "gold/building_risk/intermediate/bbl_to_bin_reference"
)

(
    unique_bbl_to_bin
    .write
    .mode("overwrite")
    .parquet(BBL_BIN_REF_PATH)
)

print("BBL -> BIN reference saved successfully")


# ==================================================
# 5. READ SAVED REFERENCE BACK
# ==================================================

unique_bbl_check = spark.read.parquet(
    BBL_BIN_REF_PATH
)


# ==================================================
# 6. RESOLVE NYC 311 USING UNIQUE BBL
# ==================================================

nyc311_bbl_resolved = (
    nyc311_df

    .filter(
        F.col("bbl").isNotNull()
        & (F.trim(F.col("bbl")) != "")
    )

    .join(
        unique_bbl_check,
        on="bbl",
        how="inner"
    )

    .withColumn(
        "match_method",
        F.lit("UNIQUE_BBL")
    )

    .withColumn(
        "match_confidence",
        F.lit("HIGH")
    )

    .withColumn(
        "resolution_status",
        F.lit("RESOLVED")
    )
)


# ==================================================
# 7. VALIDATION
# ==================================================

unique_mapping_count = (
    unique_bbl_check.count()
)

multi_bin_bbl_count = (
    bbl_bin_stats
    .filter(
        F.col("bin_count") > 1
    )
    .count()
)

resolved_311_count = (
    nyc311_bbl_resolved.count()
)

total_311_count = (
    nyc311_df.count()
)


print()
print("========================================")
print("311 -> BIN UNIQUE BBL RESOLUTION")
print("========================================")

print(
    "Unique BBL -> BIN mappings:",
    unique_mapping_count
)

print(
    "BBLs with multiple BINs:",
    multi_bin_bbl_count
)

print(
    "311 resolved by unique BBL:",
    resolved_311_count
)

print(
    "311 total:",
    total_311_count
)

print(
    "311 resolution coverage:",
    round(
        resolved_311_count
        / total_311_count
        * 100,
        2
    ),
    "%"
)

print("========================================")


# ==================================================
# 8. SAMPLE
# ==================================================

nyc311_bbl_resolved \
    .select(
        "unique_key",
        "bbl",
        "resolved_bin",
        "incident_address",
        "borough",
        "match_method",
        "match_confidence"
    ) \
    .show(
        20,
        truncate=False
    )

BBL -> BIN reference saved successfully



311 -> BIN UNIQUE BBL RESOLUTION
Unique BBL -> BIN mappings: 160915
BBLs with multiple BINs: 11760
311 resolved by unique BBL: 743355
311 total: 885306
311 resolution coverage: 83.97 %
+----------+----------+------------+---------------------+-------------+------------+----------------+
|unique_key|bbl       |resolved_bin|incident_address     |borough      |match_method|match_confidence|
+----------+----------+------------+---------------------+-------------+------------+----------------+
|67354166  |1022230005|1064786     |241 SHERMAN AVENUE   |MANHATTAN    |UNIQUE_BBL  |HIGH            |
|67354170  |1021170021|1062756     |530 WEST  159 STREET |MANHATTAN    |UNIQUE_BBL  |HIGH            |
|67354174  |3046930066|3102702     |466 ROCKAWAY PARKWAY |BROOKLYN     |UNIQUE_BBL  |HIGH            |
|67354176  |1018680001|1056059     |740 WEST END AVENUE  |MANHATTAN    |UNIQUE_BBL  |HIGH            |
|67354182  |1006150051|1011030     |31 BANK STREET       |MANHATTAN    |UNIQUE_BBL  |HIGH    

In [16]:
NYC311_BBL_RESOLVED_PATH = minio_path(
    "gold/building_risk/intermediate/311_unique_bbl_resolved"
)

(
    nyc311_bbl_resolved
    .write
    .mode("overwrite")
    .parquet(NYC311_BBL_RESOLVED_PATH)
)

print("311 unique BBL resolved saved successfully")

26/09/05 20:04:55 WARN package: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


311 unique BBL resolved saved successfully


In [17]:
from pyspark.sql import functions as F


# ==================================================
# ADD BBL STATUS TO ALL 311 RECORDS
# ==================================================

nyc311_bbl_status = (
    nyc311_df

    .join(
        bbl_bin_stats.select(
            "bbl",
            "bin_count"
        ),
        on="bbl",
        how="left"
    )

    .withColumn(
        "bbl_resolution_reason",

        F.when(
            F.col("bbl").isNull()
            | (F.trim(F.col("bbl")) == ""),
            F.lit("MISSING_BBL")
        )

        .when(
            F.col("bin_count") == 1,
            F.lit("UNIQUE_BBL")
        )

        .when(
            F.col("bin_count") > 1,
            F.lit("MULTI_BIN_BBL")
        )

        .otherwise(
            F.lit("BBL_NOT_FOUND")
        )
    )
)


# ==================================================
# SUMMARY
# ==================================================

(
    nyc311_bbl_status

    .groupBy(
        "bbl_resolution_reason"
    )

    .count()

    .orderBy(
        F.col("count").desc()
    )

    .show(
        truncate=False
    )
)

+---------------------+------+
|bbl_resolution_reason|count |
+---------------------+------+
|UNIQUE_BBL           |743355|
|MULTI_BIN_BBL        |99607 |
|BBL_NOT_FOUND        |39677 |
|MISSING_BBL          |2667  |
+---------------------+------+



In [18]:
nyc311_unresolved = (
    nyc311_bbl_status

    .filter(
        F.col("bbl_resolution_reason")
        !=
        "UNIQUE_BBL"
    )
)


NYC311_UNRESOLVED_PATH = minio_path(
    "gold/building_risk/intermediate/311_unresolved_after_bbl"
)

(
    nyc311_unresolved
    .write
    .mode("overwrite")
    .parquet(NYC311_UNRESOLVED_PATH)
)

print(
    "311 unresolved after BBL stage:",
    nyc311_unresolved.count()
)

311 unresolved after BBL stage: 141951


In [19]:
from pyspark.sql import functions as F


# ==================================================
# ADDRESS NORMALIZATION
# ==================================================

def normalize_full_address(column):

    value = F.upper(
        F.trim(column)
    )

    # Remove punctuation / special characters
    value = F.regexp_replace(
        value,
        r"[^A-Z0-9 ]",
        " "
    )

    # Normalize directions
    value = F.regexp_replace(value, r"\bWEST\b", "W")
    value = F.regexp_replace(value, r"\bEAST\b", "E")
    value = F.regexp_replace(value, r"\bNORTH\b", "N")
    value = F.regexp_replace(value, r"\bSOUTH\b", "S")

    # Normalize street types
    value = F.regexp_replace(value, r"\bSTREET\b", "ST")
    value = F.regexp_replace(value, r"\bAVENUE\b", "AVE")
    value = F.regexp_replace(value, r"\bBOULEVARD\b", "BLVD")
    value = F.regexp_replace(value, r"\bROAD\b", "RD")
    value = F.regexp_replace(value, r"\bDRIVE\b", "DR")
    value = F.regexp_replace(value, r"\bPLACE\b", "PL")
    value = F.regexp_replace(value, r"\bCOURT\b", "CT")
    value = F.regexp_replace(value, r"\bLANE\b", "LN")
    value = F.regexp_replace(value, r"\bPARKWAY\b", "PKWY")
    value = F.regexp_replace(value, r"\bTERRACE\b", "TER")

    # Remove duplicate spaces
    value = F.trim(
        F.regexp_replace(
            value,
            r"\s+",
            " "
        )
    )

    return value


def normalize_borough(column):

    value = F.upper(
        F.trim(column)
    )

    return (
        F.when(
            value.isin("1", "MN", "MANHATTAN"),
            "MANHATTAN"
        )
        .when(
            value.isin("2", "BX", "BRONX"),
            "BRONX"
        )
        .when(
            value.isin("3", "BK", "BROOKLYN"),
            "BROOKLYN"
        )
        .when(
            value.isin("4", "QN", "QUEENS"),
            "QUEENS"
        )
        .when(
            value.isin("5", "SI", "STATEN ISLAND"),
            "STATEN ISLAND"
        )
        .otherwise(value)
    )


# ==================================================
# BUILD BIN + BBL + ADDRESS REFERENCE
# ==================================================

empty_string_array = F.expr(
    "CAST(array() AS array<string>)"
)


building_address_ref = (
    building_identity_df

    # All known BBLs for each BIN
    .withColumn(
        "all_bbls",
        F.array_distinct(
            F.concat(
                F.array(
                    F.col("current_bbl"),
                    F.col("resolved_bbl")
                ),
                F.coalesce(
                    F.col("bbl_aliases"),
                    empty_string_array
                )
            )
        )
    )

    # All known addresses for each BIN
    .withColumn(
        "all_addresses",
        F.array_distinct(
            F.concat(
                F.array(
                    F.col("current_address")
                ),
                F.coalesce(
                    F.col("address_aliases"),
                    empty_string_array
                )
            )
        )
    )

    .select(
        "bin",
        "borough",
        F.explode_outer(
            "all_bbls"
        ).alias("candidate_bbl"),
        F.explode_outer(
            "all_addresses"
        ).alias("candidate_address")
    )

    .filter(
        F.col("candidate_bbl").isNotNull()
        &
        F.col("candidate_address").isNotNull()
        &
        (F.trim(F.col("candidate_address")) != "")
    )

    .withColumn(
        "normalized_address",
        normalize_full_address(
            F.col("candidate_address")
        )
    )

    .withColumn(
        "normalized_borough",
        normalize_borough(
            F.col("borough")
        )
    )

    .dropDuplicates(
        [
            "bin",
            "candidate_bbl",
            "normalized_address",
            "normalized_borough"
        ]
    )
)


# ==================================================
# ONLY 311 RECORDS WITH MULTI-BIN BBL
# ==================================================

nyc311_multi_bin = (
    nyc311_unresolved

    .filter(
        F.col("bbl_resolution_reason")
        ==
        "MULTI_BIN_BBL"
    )

    .filter(
        F.col("incident_address").isNotNull()
        &
        (F.trim(F.col("incident_address")) != "")
    )

    .withColumn(
        "normalized_address",
        normalize_full_address(
            F.col("incident_address")
        )
    )

    .withColumn(
        "normalized_borough",
        normalize_borough(
            F.col("borough")
        )
    )
)


# ==================================================
# MATCH ADDRESS INSIDE SAME BBL
# ==================================================

multi_bbl_matches_raw = (
    nyc311_multi_bin.alias("n")

    .join(
        building_address_ref.alias("b"),

        (
            F.col("n.bbl")
            ==
            F.col("b.candidate_bbl")
        )
        &
        (
            F.col("n.normalized_address")
            ==
            F.col("b.normalized_address")
        )
        &
        (
            F.col("n.normalized_borough")
            ==
            F.col("b.normalized_borough")
        ),

        how="inner"
    )

    .select(
        F.col("n.unique_key").alias("unique_key"),
        F.col("b.bin").alias("candidate_bin")
    )

    .dropDuplicates()
)


# ==================================================
# COUNT HOW MANY BIN CANDIDATES EACH 311 RECORD HAS
# ==================================================

multi_bbl_match_summary = (
    multi_bbl_matches_raw

    .groupBy("unique_key")

    .agg(
        F.countDistinct(
            "candidate_bin"
        ).alias("candidate_bin_count"),

        F.first(
            "candidate_bin"
        ).alias("resolved_bin")
    )
)


# ==================================================
# SAFE MATCHES ONLY
# EXACTLY ONE BIN
# ==================================================

multi_bbl_resolved_keys = (
    multi_bbl_match_summary

    .filter(
        F.col("candidate_bin_count") == 1
    )

    .select(
        "unique_key",
        "resolved_bin"
    )
)


# ==================================================
# FULL RESOLVED 311 RECORDS
# ==================================================

nyc311_multi_bbl_resolved = (
    nyc311_unresolved

    .join(
        multi_bbl_resolved_keys,
        on="unique_key",
        how="inner"
    )

    .withColumn(
        "match_method",
        F.lit("BBL_EXACT_ADDRESS")
    )

    .withColumn(
        "match_confidence",
        F.lit("HIGH")
    )

    .withColumn(
        "resolution_status",
        F.lit("RESOLVED")
    )
)


# ==================================================
# REMOVE NEWLY RESOLVED RECORDS
# ==================================================

nyc311_still_unresolved = (
    nyc311_unresolved

    .join(
        multi_bbl_resolved_keys.select(
            "unique_key"
        ),
        on="unique_key",
        how="left_anti"
    )
)


# ==================================================
# SAVE RESULTS
# ==================================================

MULTI_BBL_RESOLVED_PATH = minio_path(
    "gold/building_risk/intermediate/311_multi_bbl_address_resolved"
)

STILL_UNRESOLVED_311_PATH = minio_path(
    "gold/building_risk/intermediate/311_unresolved_after_multi_bbl_address"
)


(
    nyc311_multi_bbl_resolved
    .write
    .mode("overwrite")
    .parquet(MULTI_BBL_RESOLVED_PATH)
)


(
    nyc311_still_unresolved
    .write
    .mode("overwrite")
    .parquet(STILL_UNRESOLVED_311_PATH)
)


print(
    "311 multi-BBL address stage saved successfully"
)

AnalysisException: [UNSUPPORTED_GENERATOR.MULTI_GENERATOR] The generator is not supported: only one generator allowed per SELECT clause but found 2: "generatorouter(explode(all_bbls))", "generatorouter(explode(all_addresses))".

In [20]:
from pyspark.sql import functions as F


# ==================================================
# ADDRESS NORMALIZATION
# ==================================================

def normalize_full_address(column):

    value = F.upper(
        F.trim(column)
    )

    value = F.regexp_replace(
        value,
        r"[^A-Z0-9 ]",
        " "
    )

    # Directions
    value = F.regexp_replace(value, r"\bWEST\b", "W")
    value = F.regexp_replace(value, r"\bEAST\b", "E")
    value = F.regexp_replace(value, r"\bNORTH\b", "N")
    value = F.regexp_replace(value, r"\bSOUTH\b", "S")

    # Street types
    value = F.regexp_replace(value, r"\bSTREET\b", "ST")
    value = F.regexp_replace(value, r"\bAVENUE\b", "AVE")
    value = F.regexp_replace(value, r"\bBOULEVARD\b", "BLVD")
    value = F.regexp_replace(value, r"\bROAD\b", "RD")
    value = F.regexp_replace(value, r"\bDRIVE\b", "DR")
    value = F.regexp_replace(value, r"\bPLACE\b", "PL")
    value = F.regexp_replace(value, r"\bCOURT\b", "CT")
    value = F.regexp_replace(value, r"\bLANE\b", "LN")
    value = F.regexp_replace(value, r"\bPARKWAY\b", "PKWY")
    value = F.regexp_replace(value, r"\bTERRACE\b", "TER")

    value = F.trim(
        F.regexp_replace(
            value,
            r"\s+",
            " "
        )
    )

    return value


def normalize_borough(column):

    value = F.upper(
        F.trim(column)
    )

    return (
        F.when(
            value.isin("1", "MN", "MANHATTAN"),
            "MANHATTAN"
        )
        .when(
            value.isin("2", "BX", "BRONX"),
            "BRONX"
        )
        .when(
            value.isin("3", "BK", "BROOKLYN"),
            "BROOKLYN"
        )
        .when(
            value.isin("4", "QN", "QUEENS"),
            "QUEENS"
        )
        .when(
            value.isin("5", "SI", "STATEN ISLAND"),
            "STATEN ISLAND"
        )
        .otherwise(value)
    )


# ==================================================
# EMPTY ARRAY FOR NULL ALIASES
# ==================================================

empty_string_array = F.expr(
    "CAST(array() AS array<string>)"
)


# ==================================================
# BUILD BIN + BBL + ADDRESS REFERENCE
# ==================================================

building_address_ref = (
    building_identity_df

    # All known BBLs
    .withColumn(
        "all_bbls",
        F.array_distinct(
            F.concat(
                F.array(
                    F.col("current_bbl"),
                    F.col("resolved_bbl")
                ),
                F.coalesce(
                    F.col("bbl_aliases"),
                    empty_string_array
                )
            )
        )
    )

    # All known addresses
    .withColumn(
        "all_addresses",
        F.array_distinct(
            F.concat(
                F.array(
                    F.col("current_address")
                ),
                F.coalesce(
                    F.col("address_aliases"),
                    empty_string_array
                )
            )
        )
    )

    # IMPORTANT:
    # explode BBL first
    .withColumn(
        "candidate_bbl",
        F.explode_outer(
            F.col("all_bbls")
        )
    )

    # Then explode address separately
    .withColumn(
        "candidate_address",
        F.explode_outer(
            F.col("all_addresses")
        )
    )

    .select(
        "bin",
        "borough",
        "candidate_bbl",
        "candidate_address"
    )

    .filter(
        F.col("candidate_bbl").isNotNull()
        &
        F.col("candidate_address").isNotNull()
        &
        (F.trim(F.col("candidate_address")) != "")
    )

    .withColumn(
        "normalized_address",
        normalize_full_address(
            F.col("candidate_address")
        )
    )

    .withColumn(
        "normalized_borough",
        normalize_borough(
            F.col("borough")
        )
    )

    .dropDuplicates(
        [
            "bin",
            "candidate_bbl",
            "normalized_address",
            "normalized_borough"
        ]
    )
)


# ==================================================
# ONLY 311 RECORDS WITH MULTI-BIN BBL
# ==================================================

nyc311_multi_bin = (
    nyc311_unresolved

    .filter(
        F.col("bbl_resolution_reason")
        ==
        "MULTI_BIN_BBL"
    )

    .filter(
        F.col("incident_address").isNotNull()
        &
        (F.trim(F.col("incident_address")) != "")
    )

    .withColumn(
        "normalized_address",
        normalize_full_address(
            F.col("incident_address")
        )
    )

    .withColumn(
        "normalized_borough",
        normalize_borough(
            F.col("borough")
        )
    )
)


# ==================================================
# MATCH ADDRESS INSIDE SAME BBL
# ==================================================

multi_bbl_matches_raw = (
    nyc311_multi_bin.alias("n")

    .join(
        building_address_ref.alias("b"),

        (
            F.col("n.bbl")
            ==
            F.col("b.candidate_bbl")
        )
        &
        (
            F.col("n.normalized_address")
            ==
            F.col("b.normalized_address")
        )
        &
        (
            F.col("n.normalized_borough")
            ==
            F.col("b.normalized_borough")
        ),

        how="inner"
    )

    .select(
        F.col("n.unique_key").alias("unique_key"),
        F.col("b.bin").alias("candidate_bin")
    )

    .dropDuplicates()
)


# ==================================================
# COUNT CANDIDATE BINS PER 311 COMPLAINT
# ==================================================

multi_bbl_match_summary = (
    multi_bbl_matches_raw

    .groupBy("unique_key")

    .agg(
        F.countDistinct(
            "candidate_bin"
        ).alias("candidate_bin_count"),

        F.first(
            "candidate_bin"
        ).alias("resolved_bin")
    )
)


# ==================================================
# SAFE MATCHES ONLY
# ==================================================

multi_bbl_resolved_keys = (
    multi_bbl_match_summary

    .filter(
        F.col("candidate_bin_count") == 1
    )

    .select(
        "unique_key",
        "resolved_bin"
    )
)


# ==================================================
# FULL RESOLVED 311 RECORDS
# ==================================================

nyc311_multi_bbl_resolved = (
    nyc311_unresolved

    .join(
        multi_bbl_resolved_keys,
        on="unique_key",
        how="inner"
    )

    .withColumn(
        "match_method",
        F.lit("BBL_EXACT_ADDRESS")
    )

    .withColumn(
        "match_confidence",
        F.lit("HIGH")
    )

    .withColumn(
        "resolution_status",
        F.lit("RESOLVED")
    )
)


# ==================================================
# STILL UNRESOLVED
# ==================================================

nyc311_still_unresolved = (
    nyc311_unresolved

    .join(
        multi_bbl_resolved_keys.select(
            "unique_key"
        ),
        on="unique_key",
        how="left_anti"
    )
)


# ==================================================
# SAVE RESULTS
# ==================================================

MULTI_BBL_RESOLVED_PATH = minio_path(
    "gold/building_risk/intermediate/311_multi_bbl_address_resolved"
)

STILL_UNRESOLVED_311_PATH = minio_path(
    "gold/building_risk/intermediate/311_unresolved_after_multi_bbl_address"
)


(
    nyc311_multi_bbl_resolved
    .write
    .mode("overwrite")
    .parquet(MULTI_BBL_RESOLVED_PATH)
)


(
    nyc311_still_unresolved
    .write
    .mode("overwrite")
    .parquet(STILL_UNRESOLVED_311_PATH)
)


print(
    "311 multi-BBL address stage saved successfully"
)

311 multi-BBL address stage saved successfully


In [21]:
multi_resolved_check = spark.read.parquet(
    MULTI_BBL_RESOLVED_PATH
)

still_unresolved_check = spark.read.parquet(
    STILL_UNRESOLVED_311_PATH
)


print(
    "Resolved by multi-BBL exact address:",
    multi_resolved_check.count()
)

print(
    "Still unresolved:",
    still_unresolved_check.count()
)


print("\nRemaining unresolved reasons:")

(
    still_unresolved_check
    .groupBy(
        "bbl_resolution_reason"
    )
    .count()
    .orderBy(
        F.col("count").desc()
    )
    .show(truncate=False)
)


print("\nAmbiguous address matches:")

(
    multi_bbl_match_summary
    .filter(
        F.col("candidate_bin_count") > 1
    )
    .groupBy(
        "candidate_bin_count"
    )
    .count()
    .orderBy(
        "candidate_bin_count"
    )
    .show()
)

Resolved by multi-BBL exact address: 95658
Still unresolved: 46293

Remaining unresolved reasons:
+---------------------+-----+
|bbl_resolution_reason|count|
+---------------------+-----+
|BBL_NOT_FOUND        |39677|
|MULTI_BIN_BBL        |3949 |
|MISSING_BBL          |2667 |
+---------------------+-----+


Ambiguous address matches:


+-------------------+-----+
|candidate_bin_count|count|
+-------------------+-----+
|                  2| 1535|
|                  3|    1|
+-------------------+-----+



In [22]:
from pyspark.sql import functions as F


# ==================================================
# NORMALIZATION
# ==================================================

def normalize_full_address(column):

    value = F.upper(F.trim(column))

    value = F.regexp_replace(
        value,
        r"[^A-Z0-9 ]",
        " "
    )

    # Directions
    value = F.regexp_replace(value, r"\bWEST\b", "W")
    value = F.regexp_replace(value, r"\bEAST\b", "E")
    value = F.regexp_replace(value, r"\bNORTH\b", "N")
    value = F.regexp_replace(value, r"\bSOUTH\b", "S")

    # Street types
    value = F.regexp_replace(value, r"\bSTREET\b", "ST")
    value = F.regexp_replace(value, r"\bAVENUE\b", "AVE")
    value = F.regexp_replace(value, r"\bBOULEVARD\b", "BLVD")
    value = F.regexp_replace(value, r"\bROAD\b", "RD")
    value = F.regexp_replace(value, r"\bDRIVE\b", "DR")
    value = F.regexp_replace(value, r"\bPLACE\b", "PL")
    value = F.regexp_replace(value, r"\bCOURT\b", "CT")
    value = F.regexp_replace(value, r"\bLANE\b", "LN")
    value = F.regexp_replace(value, r"\bPARKWAY\b", "PKWY")
    value = F.regexp_replace(value, r"\bTERRACE\b", "TER")

    value = F.trim(
        F.regexp_replace(
            value,
            r"\s+",
            " "
        )
    )

    return value


def normalize_borough(column):

    value = F.upper(F.trim(column))

    return (
        F.when(value.isin("1", "MN", "MANHATTAN"), "MANHATTAN")
        .when(value.isin("2", "BX", "BRONX"), "BRONX")
        .when(value.isin("3", "BK", "BROOKLYN"), "BROOKLYN")
        .when(value.isin("4", "QN", "QUEENS"), "QUEENS")
        .when(
            value.isin("5", "SI", "STATEN ISLAND"),
            "STATEN ISLAND"
        )
        .otherwise(value)
    )


# ==================================================
# BUILD GLOBAL ADDRESS -> BIN REFERENCE
# ==================================================

empty_string_array = F.expr(
    "CAST(array() AS array<string>)"
)


building_global_address_ref = (
    building_identity_df

    .withColumn(
        "all_addresses",
        F.array_distinct(
            F.concat(
                F.array(
                    F.col("current_address")
                ),
                F.coalesce(
                    F.col("address_aliases"),
                    empty_string_array
                )
            )
        )
    )

    .withColumn(
        "candidate_address",
        F.explode_outer(
            F.col("all_addresses")
        )
    )

    .filter(
        F.col("candidate_address").isNotNull()
        &
        (F.trim(F.col("candidate_address")) != "")
        &
        F.col("borough").isNotNull()
    )

    .select(
        "bin",
        normalize_full_address(
            F.col("candidate_address")
        ).alias("normalized_address"),

        normalize_borough(
            F.col("borough")
        ).alias("normalized_borough")
    )

    .filter(
        F.col("normalized_address").isNotNull()
        &
        (F.col("normalized_address") != "")
        &
        F.col("normalized_borough").isNotNull()
    )

    .dropDuplicates(
        [
            "bin",
            "normalized_address",
            "normalized_borough"
        ]
    )
)


# ==================================================
# COUNT BINS PER ADDRESS + BOROUGH
# ==================================================

global_address_stats = (
    building_global_address_ref

    .groupBy(
        "normalized_address",
        "normalized_borough"
    )

    .agg(
        F.countDistinct(
            "bin"
        ).alias("candidate_bin_count"),

        F.first(
            "bin"
        ).alias("resolved_bin")
    )
)


# ==================================================
# KEEP ONLY UNIQUE ADDRESS -> BIN
# ==================================================

unique_address_to_bin = (
    global_address_stats

    .filter(
        F.col("candidate_bin_count") == 1
    )

    .select(
        "normalized_address",
        "normalized_borough",
        "resolved_bin"
    )
)


# ==================================================
# READ PHYSICAL UNRESOLVED DATASET
# ==================================================

nyc311_address_unresolved = (
    spark.read.parquet(
        STILL_UNRESOLVED_311_PATH
    )

    .filter(
        F.col("incident_address").isNotNull()
        &
        (F.trim(F.col("incident_address")) != "")
    )

    .withColumn(
        "normalized_address",
        normalize_full_address(
            F.col("incident_address")
        )
    )

    .withColumn(
        "normalized_borough",
        normalize_borough(
            F.col("borough")
        )
    )
)


# ==================================================
# EXACT ADDRESS + BOROUGH MATCH
# ==================================================

global_address_resolved_keys = (
    nyc311_address_unresolved.alias("n")

    .join(
        F.broadcast(
            unique_address_to_bin
        ).alias("b"),

        (
            F.col("n.normalized_address")
            ==
            F.col("b.normalized_address")
        )
        &
        (
            F.col("n.normalized_borough")
            ==
            F.col("b.normalized_borough")
        ),

        how="inner"
    )

    .select(
        F.col("n.unique_key").alias("unique_key"),
        F.col("b.resolved_bin").alias("resolved_bin")
    )

    .dropDuplicates(
        ["unique_key"]
    )
)


# ==================================================
# FULL RESOLVED RECORDS
# ==================================================

nyc311_global_address_resolved = (
    spark.read.parquet(
        STILL_UNRESOLVED_311_PATH
    )

    .join(
        global_address_resolved_keys,
        on="unique_key",
        how="inner"
    )

    .withColumn(
        "match_method",
        F.lit("EXACT_ADDRESS_BOROUGH")
    )

    .withColumn(
        "match_confidence",
        F.lit("HIGH")
    )

    .withColumn(
        "resolution_status",
        F.lit("RESOLVED")
    )
)


# ==================================================
# STILL UNRESOLVED AFTER GLOBAL ADDRESS
# ==================================================

nyc311_after_global_address = (
    spark.read.parquet(
        STILL_UNRESOLVED_311_PATH
    )

    .join(
        global_address_resolved_keys.select(
            "unique_key"
        ),
        on="unique_key",
        how="left_anti"
    )
)


# ==================================================
# SAVE
# ==================================================

GLOBAL_ADDRESS_RESOLVED_PATH = minio_path(
    "gold/building_risk/intermediate/311_global_address_resolved"
)

UNRESOLVED_AFTER_ADDRESS_PATH = minio_path(
    "gold/building_risk/intermediate/311_unresolved_after_address"
)


(
    nyc311_global_address_resolved
    .write
    .mode("overwrite")
    .parquet(GLOBAL_ADDRESS_RESOLVED_PATH)
)


(
    nyc311_after_global_address
    .write
    .mode("overwrite")
    .parquet(UNRESOLVED_AFTER_ADDRESS_PATH)
)


print(
    "311 global exact-address stage saved successfully"
)

311 global exact-address stage saved successfully


In [23]:
global_address_resolved_check = spark.read.parquet(
    GLOBAL_ADDRESS_RESOLVED_PATH
)

unresolved_after_address_check = spark.read.parquet(
    UNRESOLVED_AFTER_ADDRESS_PATH
)


print(
    "Resolved by global exact address:",
    global_address_resolved_check.count()
)

print(
    "Still unresolved:",
    unresolved_after_address_check.count()
)


print("\nRemaining unresolved reasons:")

(
    unresolved_after_address_check
    .groupBy(
        "bbl_resolution_reason"
    )
    .count()
    .orderBy(
        F.col("count").desc()
    )
    .show(truncate=False)
)


print("\nUnique address -> BIN mappings:")

print(
    unique_address_to_bin.count()
)


print("\nAmbiguous building addresses:")

(
    global_address_stats
    .filter(
        F.col("candidate_bin_count") > 1
    )
    .groupBy(
        "candidate_bin_count"
    )
    .count()
    .orderBy(
        "candidate_bin_count"
    )
    .show()
)

Resolved by global exact address: 446
Still unresolved: 45847

Remaining unresolved reasons:
+---------------------+-----+
|bbl_resolution_reason|count|
+---------------------+-----+
|BBL_NOT_FOUND        |39634|
|MULTI_BIN_BBL        |3927 |
|MISSING_BBL          |2286 |
+---------------------+-----+


Unique address -> BIN mappings:


202180

Ambiguous building addresses:


+-------------------+-----+
|candidate_bin_count|count|
+-------------------+-----+
|                  2|  238|
|                  3|    2|
+-------------------+-----+



In [24]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window


# ==================================================
# READ REMAINING 311
# ==================================================

geo_source = spark.read.parquet(
    UNRESOLVED_AFTER_ADDRESS_PATH
)


# ==================================================
# NORMALIZE BOROUGH
# ==================================================

def normalize_borough(column):

    value = F.upper(F.trim(column))

    return (
        F.when(value.isin("1", "MN", "MANHATTAN"), "MANHATTAN")
        .when(value.isin("2", "BX", "BRONX"), "BRONX")
        .when(value.isin("3", "BK", "BROOKLYN"), "BROOKLYN")
        .when(value.isin("4", "QN", "QUEENS"), "QUEENS")
        .when(
            value.isin("5", "SI", "STATEN ISLAND"),
            "STATEN ISLAND"
        )
        .otherwise(value)
    )


# ==================================================
# CHECK COORDINATE AVAILABILITY
# ==================================================

geo_source = (
    geo_source
    .withColumn(
        "has_coordinates",
        F.when(
            F.col("latitude").isNotNull()
            & F.col("longitude").isNotNull(),
            1
        ).otherwise(0)
    )
)


print("=== COORDINATE AVAILABILITY ===")

(
    geo_source
    .groupBy(
        "bbl_resolution_reason",
        "has_coordinates"
    )
    .count()
    .orderBy(
        "bbl_resolution_reason",
        "has_coordinates"
    )
    .show(truncate=False)
)


# ==================================================
# ONLY RECORDS WITH COORDINATES
# ==================================================

geo_source_valid = (
    geo_source

    .filter(
        F.col("latitude").isNotNull()
        & F.col("longitude").isNotNull()
    )

    .withColumn(
        "normalized_borough",
        normalize_borough(
            F.col("borough")
        )
    )

    # Grid size ~0.001 degree
    .withColumn(
        "grid_lat",
        F.floor(
            F.col("latitude") * 1000
        )
    )

    .withColumn(
        "grid_lon",
        F.floor(
            F.col("longitude") * 1000
        )
    )
)


# ==================================================
# BUILD BIN GEO REFERENCE
# ==================================================

building_geo_ref = (
    building_identity_df

    .filter(
        F.col("latitude").isNotNull()
        & F.col("longitude").isNotNull()
    )

    .select(
        F.col("bin").alias("candidate_bin"),
        F.col("resolved_bbl").alias("candidate_bbl"),
        F.col("current_address").alias("candidate_address"),

        normalize_borough(
            F.col("borough")
        ).alias("normalized_borough"),

        F.col("latitude").alias("candidate_latitude"),
        F.col("longitude").alias("candidate_longitude")
    )

    .withColumn(
        "grid_lat",
        F.floor(
            F.col("candidate_latitude") * 1000
        )
    )

    .withColumn(
        "grid_lon",
        F.floor(
            F.col("candidate_longitude") * 1000
        )
    )
)


# ==================================================
# CREATE 9 NEIGHBORING GRID CELLS
# ==================================================

offsets = spark.createDataFrame(
    [
        (-1, -1), (-1, 0), (-1, 1),
        (0, -1),  (0, 0),  (0, 1),
        (1, -1),  (1, 0),  (1, 1)
    ],
    ["lat_offset", "lon_offset"]
)


geo_source_expanded = (
    geo_source_valid

    .crossJoin(
        F.broadcast(offsets)
    )

    .withColumn(
        "join_grid_lat",
        F.col("grid_lat")
        + F.col("lat_offset")
    )

    .withColumn(
        "join_grid_lon",
        F.col("grid_lon")
        + F.col("lon_offset")
    )
)


# ==================================================
# CREATE GEO CANDIDATES
# ==================================================

geo_candidates = (
    geo_source_expanded.alias("s")

    .join(
        building_geo_ref.alias("b"),

        (
            F.col("s.normalized_borough")
            ==
            F.col("b.normalized_borough")
        )
        &
        (
            F.col("s.join_grid_lat")
            ==
            F.col("b.grid_lat")
        )
        &
        (
            F.col("s.join_grid_lon")
            ==
            F.col("b.grid_lon")
        ),

        how="inner"
    )

    .select(
        F.col("s.unique_key").alias("unique_key"),
        F.col("s.bbl_resolution_reason").alias(
            "bbl_resolution_reason"
        ),

        F.col("s.latitude").alias("source_latitude"),
        F.col("s.longitude").alias("source_longitude"),

        F.col("b.candidate_bin"),
        F.col("b.candidate_bbl"),
        F.col("b.candidate_address"),
        F.col("b.candidate_latitude"),
        F.col("b.candidate_longitude")
    )
)


# ==================================================
# HAVERSINE DISTANCE IN METERS
# ==================================================

lat1 = F.radians(
    F.col("source_latitude")
)

lon1 = F.radians(
    F.col("source_longitude")
)

lat2 = F.radians(
    F.col("candidate_latitude")
)

lon2 = F.radians(
    F.col("candidate_longitude")
)


a = (
    F.pow(
        F.sin(
            (lat2 - lat1) / 2
        ),
        2
    )
    +
    F.cos(lat1)
    *
    F.cos(lat2)
    *
    F.pow(
        F.sin(
            (lon2 - lon1) / 2
        ),
        2
    )
)


geo_candidates = (
    geo_candidates

    .withColumn(
        "distance_m",
        2
        * F.lit(6371000.0)
        * F.asin(
            F.sqrt(a)
        )
    )

    # For now profile candidates up to 100 meters
    .filter(
        F.col("distance_m") <= 100
    )

    .dropDuplicates(
        [
            "unique_key",
            "candidate_bin"
        ]
    )
)


# ==================================================
# FIND NEAREST / SECOND NEAREST
# ==================================================

distance_window = (
    Window
    .partitionBy("unique_key")
    .orderBy(
        F.col("distance_m").asc()
    )
)


ranked_geo_candidates = (
    geo_candidates
    .withColumn(
        "geo_rank",
        F.row_number().over(
            distance_window
        )
    )
)


geo_profile = (
    ranked_geo_candidates

    .groupBy(
        "unique_key"
    )

    .agg(
        F.countDistinct(
            "candidate_bin"
        ).alias("candidate_count_100m"),

        F.max(
            F.when(
                F.col("geo_rank") == 1,
                F.col("candidate_bin")
            )
        ).alias("nearest_bin"),

        F.max(
            F.when(
                F.col("geo_rank") == 1,
                F.col("distance_m")
            )
        ).alias("nearest_distance_m"),

        F.max(
            F.when(
                F.col("geo_rank") == 2,
                F.col("distance_m")
            )
        ).alias("second_distance_m")
    )
)


# ==================================================
# SAVE PROFILE
# ==================================================

GEO_PROFILE_PATH = minio_path(
    "gold/building_risk/intermediate/311_geo_profile"
)


(
    geo_profile
    .write
    .mode("overwrite")
    .parquet(GEO_PROFILE_PATH)
)


print("311 GEO profile saved successfully")

=== COORDINATE AVAILABILITY ===
+---------------------+---------------+-----+
|bbl_resolution_reason|has_coordinates|count|
+---------------------+---------------+-----+
|BBL_NOT_FOUND        |1              |39634|
|MISSING_BBL          |0              |42   |
|MISSING_BBL          |1              |2244 |
|MULTI_BIN_BBL        |1              |3927 |
+---------------------+---------------+-----+



311 GEO profile saved successfully


In [25]:
geo_profile_check = spark.read.parquet(
    GEO_PROFILE_PATH
)


print(
    "Remaining 311:",
    geo_source.count()
)

print(
    "Remaining with coordinates:",
    geo_source_valid.count()
)

print(
    "311 with GEO candidate <= 100m:",
    geo_profile_check.count()
)


print("\nNearest distance distribution:")

(
    geo_profile_check

    .withColumn(
        "distance_bucket",

        F.when(
            F.col("nearest_distance_m") <= 10,
            "00-10m"
        )
        .when(
            F.col("nearest_distance_m") <= 25,
            "11-25m"
        )
        .when(
            F.col("nearest_distance_m") <= 50,
            "26-50m"
        )
        .otherwise(
            "51-100m"
        )
    )

    .groupBy(
        "distance_bucket"
    )

    .count()

    .orderBy(
        "distance_bucket"
    )

    .show(truncate=False)
)


print("\nNumber of candidates within 100m:")

(
    geo_profile_check

    .groupBy(
        "candidate_count_100m"
    )

    .count()

    .orderBy(
        "candidate_count_100m"
    )

    .show(20, truncate=False)
)

Remaining 311: 45847
Remaining with coordinates: 45805
311 with GEO candidate <= 100m: 44837

Nearest distance distribution:
+---------------+-----+
|distance_bucket|count|
+---------------+-----+
|00-10m         |27932|
|11-25m         |7924 |
|26-50m         |4772 |
|51-100m        |4209 |
+---------------+-----+


Number of candidates within 100m:
+--------------------+-----+
|candidate_count_100m|count|
+--------------------+-----+
|1                   |1165 |
|2                   |1140 |
|3                   |1353 |
|4                   |1606 |
|5                   |1405 |
|6                   |1188 |
|7                   |1555 |
|8                   |1103 |
|9                   |1241 |
|10                  |1137 |
|11                  |1180 |
|12                  |1047 |
|13                  |1044 |
|14                  |1061 |
|15                  |1207 |
|16                  |1008 |
|17                  |1038 |
|18                  |911  |
|19                  |1194 |
|20      

26/09/07 10:25:57 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 7617931 ms exceeds timeout 120000 ms
26/09/07 10:25:57 WARN SparkContext: Killing executors is not supported by current scheduler.
26/09/07 10:26:00 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:322)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:117)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$driverEndpoint(BlockManagerMasterEndpoint.scala:116)
	at org.apache.spark.storage.

In [1]:
import sys

from pyspark.sql import SparkSession
from pyspark.sql import functions as F


# ==================================================
# PROJECT CONFIG
# ==================================================

PROJECT_ROOT = "/workspace/nyc-building-risk"
COMMON_PATH = f"{PROJECT_ROOT}/spark/common"

if COMMON_PATH not in sys.path:
    sys.path.insert(0, COMMON_PATH)


# ==================================================
# IMPORT PROJECT HELPERS
# ==================================================

from minio_config import configure_minio, minio_path


# ==================================================
# CREATE / GET SPARK SESSION
# ==================================================

spark = (
    SparkSession.builder
    .appName("NYC Building Risk - Building Risk Features")
    .master("local[2]")
    .config("spark.driver.memory", "2g")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.session.timeZone", "UTC")
    .config(
        "spark.hadoop.fs.s3a.impl",
        "org.apache.hadoop.fs.s3a.S3AFileSystem"
    )
    .getOrCreate()
)

configure_minio(spark)

spark.sparkContext.setLogLevel("WARN")

print("Spark version:", spark.version)
print("Master:", spark.sparkContext.master)
print("Test:", spark.range(1).count())

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/10 15:03:56 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 3.4.0
Master: local[2]
Test: 1


In [2]:
# ==================================================
# GOLD HEALTH CHECK
# ==================================================

dim_building = spark.read.parquet(
    minio_path("gold/data_model/dim_building")
)

fact_311 = spark.read.parquet(
    minio_path("gold/data_model/fact_311_event")
)

fact_hpd = spark.read.parquet(
    minio_path("gold/data_model/fact_hpd_violation")
)

fact_dob = spark.read.parquet(
    minio_path("gold/data_model/fact_dob_violation")
)

print("dim_building:", dim_building.count())
print("fact_311:", fact_311.count())
print("fact_hpd:", fact_hpd.count())
print("fact_dob:", fact_dob.count())

26/09/10 15:04:15 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


dim_building: 197958
fact_311: 885306
fact_hpd: 927308
fact_dob: 148688
